In [1]:
import pageindex.utils as utils
import json

with open("results/HaikuDeepSeek_R1_structure.json") as f:
    tree_str = f.read()
f.close()

In [2]:
tree = json.loads(tree_str)

print(tree.keys())
print(f"Under Structure")
print(tree['structure'][0].keys())
print("")
print(f'Title: {tree['structure'][0]['title']}')
print(f'Summary: {tree['structure'][0]['summary']}')

tree = tree['structure']

dict_keys(['doc_name', 'structure'])
Under Structure
dict_keys(['title', 'start_index', 'end_index', 'node_id', 'summary'])

Title: Introduction
Summary: # Main Points Covered in DeepSeek-R1 Document

## Core Innovation
- Proposes using pure reinforcement learning (RL) to develop reasoning capabilities in large language models (LLMs) without requiring human-annotated reasoning trajectories
- Demonstrates that RL can incentivize emergent reasoning patterns such as self-reflection, verification, and dynamic strategy adaptation

## Key Limitations Addressed
- Overcomes dependency on human-labeled reasoning demonstrations that limits scalability and introduces cognitive biases
- Eliminates constraints that force models to replicate human thought processes, allowing exploration of superior non-human reasoning pathways

## Technical Approach
- Builds on DeepSeek-V3-Base using Group Relative Policy Optimization (GRPO) as the RL framework
- Uses only correctness of final predictions as reward 

In [3]:
tree_without_text = utils.remove_fields(tree.copy(), fields=['text', 'start_index', 'end_index'])
print(tree_without_text[0].keys())

dict_keys(['title', 'node_id', 'summary'])


In [ ]:
from google import genai
from google.genai import types
GEMINI_API_KEY="DUMMY"
def generate(message: str):
    client = genai.Client(
        api_key=GEMINI_API_KEY,
    )

    model = "gemini-2.5-flash"
    # contents = [
    #     types.Content(
    #         role="user",
    #         parts=[
    #             types.Part.from_text(text=message),
    #         ],
    #     ),
    # ]
    # generate_content_config = types.GenerateContentConfig(
    #     thinking_config=types.ThinkingConfig(
    #         thinking_budget=0,
    #     ),
    # )

    # for chunk in client.models.generate_content_stream(
    #     model=model,
    #     contents=contents,
    #     config=generate_content_config,
    # ):
    #     print(chunk.text, end="")
        
    response = client.models.generate_content(
        model=model,
        contents=message,
    )

    return response.text



In [30]:
query = "What are the conclusions in this document?"
search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = generate(search_prompt)

In [31]:
print(tree_search_result)

```json
{
    "thinking": "The user is asking for the conclusions of the document. I will look for nodes with titles like 'Conclusion', 'Summary', 'Discussion', or 'Key Findings', as these sections typically summarize the main points, implications, limitations, and future work of a document.",
    "node_list": [
        "0011",
        "0012",
        "0013",
        "0053",
        "0054",
        "0055"
    ]
}
```


In [32]:
test = tree_search_result.replace('\\\"', "")
test = test.replace("```json", "")
test = test.replace("```", "")

In [33]:
print(test)


{
    "thinking": "The user is asking for the conclusions of the document. I will look for nodes with titles like 'Conclusion', 'Summary', 'Discussion', or 'Key Findings', as these sections typically summarize the main points, implications, limitations, and future work of a document.",
    "node_list": [
        "0011",
        "0012",
        "0013",
        "0053",
        "0054",
        "0055"
    ]
}



In [34]:
node_map = utils.create_node_mapping(tree)
print(node_map['0000'].keys())

dict_keys(['title', 'start_index', 'end_index', 'node_id', 'summary'])


In [35]:
# parse the JSON string
tree_search_result_json = json.loads(test)
print(tree_search_result_json['node_list'])

['0011', '0012', '0013', '0053', '0054', '0055']


In [36]:
print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

Reasoning Process:
The user is asking for the conclusions of the document. I will look for nodes with titles like
'Conclusion', 'Summary', 'Discussion', or 'Key Findings', as these sections typically summarize the
main points, implications, limitations, and future work of a document.


In [37]:
print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']} \t Title: {node['title']}")


Retrieved Nodes:
Node ID: 0011 	 Title: Ethics and Safety Statement
Node ID: 0012 	 Title: Conclusion, Limitation, and Future Work
Node ID: 0013 	 Title: Author List
Node ID: 0053 	 Title: Discussion
Node ID: 0054 	 Title: Key Findings
Node ID: 0055 	 Title: Unsuccessful Attempts


In [38]:
node_list = json.loads(test)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

KeyError: 'text'